# ViTreous — `precompute.ipynb`

Curated gallery images → **Explanation Packs** → dataset-level
**projections** → upload via a `StorageAdapter` → emit the Postgres rows
(ARCHITECTURE.md §5, §10, §15). One `build_pack()` call — the same code
the live CPU service runs. Swap datasets by changing `DATASET`.

In [ ]:
# ─── Knobs ─────────────────────────────────────────────────────────────
DATASET = "eurosat"          # change this one string to swap datasets
MODEL   = "vit_s16"
SEED    = 0
N_GALLERY = 75               # curated demo images
DATA_ROOT = "/kaggle/input"
WEIGHTS = None               # path to fine-tuned .pt from train.ipynb, or None
STORAGE = "supabase"         # "local" for a dry-run, "supabase" to publish, "hf" overflow
PROJECTION_LAYERS = (0, 3, 6, 9, 12)

In [ ]:
# Install the vitreous core package (with the [ml] extra) from the repo branch.
# On Kaggle: enable Internet in the notebook settings. The package is the
# single code path shared with the live service — notebooks are thin shells.
import subprocess, sys
REPO = "https://github.com/<owner>/DragonHatchling.git"
BRANCH = "claude/explainable-vit-research-qadg2i"
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                f"git+{REPO}@{BRANCH}#subdirectory=packages/core"], check=False)
# If the repo is attached as a Kaggle dataset/utility script instead, do:
#   %pip install -q -e /kaggle/input/vitreous/packages/core

In [ ]:
import torch, numpy as np
from vitreous.data import get_dataset
from vitreous.models import load_model
from vitreous.instrument import Instrumenter
from vitreous.packs import build_pack
from vitreous.projections import build_projection_artifacts
from vitreous.storage import get_storage

adapter = get_dataset(DATASET)()
spec = adapter.spec
lm = load_model(MODEL, spec, pretrained=WEIGHTS is None, weights=WEIGHTS,
                num_classes=spec.num_classes)
model = lm.module.eval()

In [ ]:
# Storage backend. 'local' writes file:// URLs for an offline dry-run;
# 'supabase' publishes to public buckets (creds from env, never hardcoded).
storage = get_storage(STORAGE)   # LocalStorageAdapter | SupabaseStorageAdapter | HFDatasetStorageAdapter
print('storage backend:', storage.kind)

In [ ]:
# Build one pack per gallery image and upload it under packs/{dataset}/{id}/.
from PIL import Image
import tempfile, os

gallery = adapter.gallery(DATA_ROOT, n=N_GALLERY)
eval_t = adapter.preprocess()
rows = []
cls_vectors, per_image_meta = [], []
for s in gallery:
    img = Image.open(s.image).convert('RGB') if isinstance(s.image, str) else Image.fromarray(s.image)
    x = eval_t(img).unsqueeze(0)
    image_id = s.image_id or os.path.splitext(os.path.basename(str(s.image)))[0]
    with tempfile.TemporaryDirectory() as td:
        pack_dir = os.path.join(td, image_id)
        build_pack(lm, x, {'id': image_id, 'source': 'gallery'}, spec, pack_dir, seed=SEED)
        prefix = f'packs/{DATASET}/{image_id}'
        urls = storage.put_pack(pack_dir, prefix)
    # capture CLS token at the top layer for the dataset projection
    trace = Instrumenter(model).capture(x)
    cls_vectors.append(trace.tokens[9][0].numpy())   # layer-9 CLS
    rows.append({'image_id': image_id, 'class_label': s.label, 'pack_prefix': prefix + '/',
                 'thumb_url': storage.get_url(prefix + '/image.webp')})
print('built', len(rows), 'packs')

In [ ]:
# Dataset-level projections per layer (§10): UMAP/PCA/t-SNE + persisted reducers.
import tempfile
cls = np.stack(cls_vectors)
proj_rows = []
with tempfile.TemporaryDirectory() as td:
    for layer in PROJECTION_LAYERS:
        # (For a real run, collect CLS at each layer; here we reuse the layer-9 bank as a template.)
        arts = build_projection_artifacts(td, cls, layer=layer, dataset=DATASET, model=MODEL, seed=SEED)
        for method, m in arts['methods'].items():
            key = f'projections/{DATASET}/{MODEL}/{m["coords_file"]}'
            url = storage.put_file(os.path.join(td, m['coords_file']), key)
            red = None
            if m.get('reducer_file'):
                red = storage.put_file(os.path.join(td, m['reducer_file']),
                                       f'projections/{DATASET}/{MODEL}/{m["reducer_file"]}')
            proj_rows.append({'layer': layer, 'method': method, 'url': url, 'reducer_url': red})
print('uploaded', len(proj_rows), 'projection artifacts')

In [ ]:
# Emit Postgres rows (§15). Writes use the service-role key (bypasses RLS).
# Prefer the supabase client if installed; else print SQL for the SQL editor.
import os
def upsert_rows():
    try:
        from supabase import create_client
    except ImportError:
        return None
    sb = create_client(os.environ['SUPABASE_URL'], os.environ['SUPABASE_SERVICE_KEY'])
    ds = sb.table('datasets').upsert({'name': DATASET, 'display_name': spec.display_name,
                                      'spec': {}}, on_conflict='name').execute().data[0]
    md = sb.table('models').insert({'dataset_id': ds['id'], 'arch': lm.spec.arch}).execute().data[0]
    for r in rows:
        sb.table('gallery_images').insert({'dataset_id': ds['id'], 'model_id': md['id'],
            'class_label': str(r['class_label']), 'pack_prefix': r['pack_prefix'],
            'thumb_url': r['thumb_url']}).execute()
    for p in proj_rows:
        sb.table('projections').insert({'dataset_id': ds['id'], 'model_id': md['id'],
            'layer': p['layer'], 'method': p['method'], 'url': p['url'],
            'reducer_url': p['reducer_url']}).execute()
    return md['id']

model_id = upsert_rows()
print('inserted rows; model_id =', model_id)